# The connection between a 2-mode coupler and a polarisation system

In this notebook we use our analytic solution to the coherent coupler in the Weierstrass notation to explore coordinate transformations. We show that it is possible to transform the coherent coupler to a canonical coordinate system in which the solutions are single valued functions, namely Kronecker theta functions. This transformation is from the coherent coupler system to nonlinear polarisation dynamics.

In [82]:
from sympy import *
from time import time

# -- Symbols --

(x, y, z, t, x0, y0, z0, z1, t0, Z, g2, g3, m, n, l, k, i, j, M, N, C, n2, T) = symbols(
    '''x, y, z, t, x0, y0, z0, z1, t0, Z, g2, g3, m, n, l, k, i, j, M, N, C, n2, T'''
)
(delta, nu, Aeff, chi, varsigma) = symbols(
    '''delta, nu, Aeff, chi, varsigma'''
)

gbar2 = Symbol('gbar2', latex_name=r'\bar{g_2}')
gbar3 = Symbol('gbar3', latex_name=r'\bar{g_3}')
gtilde2 = Symbol('gtilde2', latex_name=r'\tilde{g_2}')
gtilde3 = Symbol('gtilde3', latex_name=r'\tilde{g_3}')

# -- Functions --

esp = Function('esp') # Elementary symmetric polynomials

pw = Function('pw') # Weierstrass P function
pwp = Function('pwp') # Derivative of Weierstrass P function
zw = Function('zw') # Weierstrass Zeta function
sigma = Function('sigma') # Weierstrass Sigma function
wp = Function('wp') # Weierstrass P function
wpp = Function('\wp\'') # Derivative of Weierstrass P function
zeta = Function('zeta') # Weierstrass Zeta function

Det = Function('Det') # Unevaluated determinant

rhop = Function('\\rho\'')
Delta = Function('Delta')
rho = Function('rho')
rhohat = Function('rhohat', latex_name=r'\hat{rho}')

kappa = Function('kappa')
phi = Function('phi')
varphi = Function('varphi')
h = Function('h')
q = Function('q')
s = Function('s')
u = Function('u')
v = Function('v')
w = Function('w')
ws = Function('ws')
xi = Function('xi')
P = Function('P') # Polynomial
Q = Function('Q') # Polynomial
R = Function('R') # Polynomial

U = Function('U')
V = Function('V')
W = Function('W')
H = Function('H')


uhat = Function('uhat', latex_name=r'\hat{u}')
vhat = Function('vhat', latex_name=r'\hat{v}')
Hhat = Symbol('Hhat', latex_name=r'\hat{H}')

ubar = Function('ubar', latex_name=r'\bar{u}')
vbar = Function('vbar', latex_name=r'\bar{v}')
Hbar = Symbol('Hbar', latex_name=r'\bar{H}')
wbar = Function('wbar', latex_name=r'\bar{w}')
rhobar = Function('rhobar', latex_name=r'\bar{\rho}')

utilde = Function('utilde', latex_name=r'\tilde{u}')
vtilde = Function('vtilde', latex_name=r'\tilde{v}')
Htilde = Symbol('Htilde', latex_name=r'\tilde{H}')
htilde = Function('htilde', latex_name=r'\tilde{h}')
wtilde = Function('wtilde', latex_name=r'\tilde{w}')
rhotilde = Function('rhotilde', latex_name=r'\tilde{\rho}')

A = Function('A')
Ac = Function('Ac')
B = Function('B')
Bc = Function('Bc')

f = Function('f')


Summ = Function('Summ')

# -- Indexed Symbols --

Omega = IndexedBase('Omega')
F = IndexedBase('F')
r = IndexedBase('r')
gamma = IndexedBase('gamma')
gammabar = IndexedBase('gammabar', latex_name=r'\bar{\gamma}')
gammahat = IndexedBase('gammahat', latex_name=r'\hat{\gamma}')
gammatilde = IndexedBase('gammatilde', latex_name=r'\tilde{\gamma}')
epsilontilde = IndexedBase('epsilontilde', latex_name=r'\tilde{\epsilon}')
ebar = IndexedBase('ebar', latex_name=r'\bar{e}')
etilde = IndexedBase('etilde', latex_name=r'\tilde{e}')
mu = IndexedBase('mu')
mubar = IndexedBase('mubar', latex_name=r'\bar{\mu}')
mutilde = IndexedBase('mutilde', latex_name=r'\tilde{\mu}')
nu = IndexedBase('nu')
theta = IndexedBase('theta')
Theta = IndexedBase('Theta')
X = IndexedBase('X')
Y = IndexedBase('Y')
a = IndexedBase('a')
b = IndexedBase('b')
c = IndexedBase('c')
d = IndexedBase('d')
p = IndexedBase('p')
G = IndexedBase('G')
psi = IndexedBase('psi')
Psi = IndexedBase('Psi')
upsilon = IndexedBase('upsilon')
epsilon = IndexedBase('epsilon')
omega = IndexedBase('omega')
alpha = IndexedBase('alpha')
beta = IndexedBase('beta')
K = IndexedBase('K')
lamda = IndexedBase('lamda') # special sympy spelling
Lamda = IndexedBase('Lamda')

wild = Wild('*')


from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

# kth order derivatives of Weierstrass P
from wpk import wpk, wzk, wsk, run_tests

# The package containing mpmath expressions for Weierstrass elliptic functions
from weierstrass_modified import Weierstrass
we = Weierstrass()
from mpmath import exp as mpexp

# Numeric solutions to diff eqs
from numpy import linspace, absolute, angle, square, real, imag, conj, array as arraynp, concatenate
from numpy import vectorize as np_vectorize # not to get confused with vectorise in other packages
import scipy.integrate
import matplotlib.pyplot as plt
from random import random

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Elliptic function identities

In [3]:
sigma_p_identity = Eq(
    wp(y, g2, g3) - wp(x, g2, g3),
    sigma(x + y, g2, g3) * sigma(x - y, g2, g3) / (sigma(x, g2, g3) ** 2 * sigma(y, g2, g3) ** 2) 
)
pw_to_zw_identity = Eq(
    (wpp(z,g2,g3) - wpp(x,g2,g3))/(wp(z,g2,g3) - wp(x,g2,g3))/2,
    zeta(z + x,g2, g3) - zeta(z,g2, g3) - zeta(x,g2, g3)
)
pw_to_zw_identity_one_sided = Eq(
    wpp(z, g2, g3)/(wp(x, g2, g3) - wp(z, g2, g3)),
    2*zeta(z, g2, g3) - zeta(-x + z, g2, g3) - zeta(x + z, g2, g3)
)

zw_dlog_sigma_identity = Eq(zeta(z - x, g2, g3), Derivative(log(sigma(z - x, g2, g3)), z))
pw_to_dlog_sigma_identity = pw_to_zw_identity.subs([
    zw_dlog_sigma_identity.subs(x,-x).args,
    zw_dlog_sigma_identity.subs(x,0).args,
])
pw_to_dlog_sigma_identity_b = pw_to_dlog_sigma_identity.subs(x,-x).subs([
    (wp(-x,g2,g3), wp(x,g2,g3)), (wpp(-x,g2,g3), -wpp(x,g2,g3)), (zeta(-x, g2,g3), -zeta(x, g2,g3))
])

pw_addition_id = Eq(
    wp(x + y, g2, g3),
    -wp(x, g2, g3) - wp(y, g2, g3) + (wpp(x, g2, g3) - wpp(y, g2, g3))**2/(4*(wp(x, g2, g3) - wp(y, g2, g3))**2)
)

pwp_sigma_dbl_ratio = Eq(wpp(z,g2,g3), - sigma(2*z,g2,g3)/sigma(z,g2,g3)**4)

quasi_period_sigma_b = Eq(
    sigma(2*m*omega[3] + 2*n*omega[1] + z, g2, g3),
    (-1)**(m*n + m + n)*sigma(z, g2, g3)*
    exp(2*(m*omega[3] + n*omega[1] + z)*zeta(m*omega[3] + n*omega[1], g2, g3))
)

# See Homogenity 
# https://functions.wolfram.com/EllipticFunctions/WeierstrassP/introductions/Weierstrass/ShowAll.html
sig_scale_law = Eq(sigma(x,g2*c**4,g3*c**6), sigma(x*c,g2,g3)/c)
zw_scale_law = Eq(zeta(x,g2*c**4,g3*c**6), zeta(x*c,g2,g3)*c)
pw_scale_law = Eq(wp(x,g2*c**4,g3*c**6), wp(x*c,g2,g3)*c**2)
pwp_scale_law = Eq(wpp(x,g2*c**4,g3*c**6), wpp(x*c,g2,g3)*c**3)

omega1_scale_law_a = Eq(omega[1], f(1, g2,g3))
omega1_scale_law_b = Eq(c*omega[1], f(1, g2/c**4,g3/c**6))
omega3_scale_law_a = Eq(omega[3], f(3, g2,g3))
omega3_scale_law_b = Eq(c*omega[3], f(3, g2/c**4,g3/c**6))

sigma_p_identity
pw_to_zw_identity
pw_to_zw_identity_one_sided
zw_dlog_sigma_identity
pw_to_dlog_sigma_identity_b
pw_addition_id
pwp_sigma_dbl_ratio
quasi_period_sigma_b

sig_scale_law
zw_scale_law
pw_scale_law
pwp_scale_law
omega1_scale_law_a
omega1_scale_law_b
omega3_scale_law_a
omega3_scale_law_b

Eq(-wp(x, g2, g3) + wp(y, g2, g3), sigma(x - y, g2, g3)*sigma(x + y, g2, g3)/(sigma(x, g2, g3)**2*sigma(y, g2, g3)**2))

Eq((-\wp'(x, g2, g3) + \wp'(z, g2, g3))/(2*(-wp(x, g2, g3) + wp(z, g2, g3))), -zeta(x, g2, g3) - zeta(z, g2, g3) + zeta(x + z, g2, g3))

Eq(\wp'(z, g2, g3)/(wp(x, g2, g3) - wp(z, g2, g3)), 2*zeta(z, g2, g3) - zeta(-x + z, g2, g3) - zeta(x + z, g2, g3))

Eq(zeta(-x + z, g2, g3), Derivative(log(sigma(-x + z, g2, g3)), z))

Eq((\wp'(x, g2, g3) + \wp'(z, g2, g3))/(2*(-wp(x, g2, g3) + wp(z, g2, g3))), zeta(x, g2, g3) - Derivative(log(sigma(z, g2, g3)), z) + Derivative(log(sigma(-x + z, g2, g3)), z))

...

Eq(\wp'(x, g2*c**4, g3*c**6), \wp'(x*c, g2, g3)*c**3)

Eq(omega[1], f(1, g2, g3))

Eq(omega[1]*c, f(1, g2/c**4, g3/c**6))

Eq(omega[3], f(3, g2, g3))

Eq(omega[3]*c, f(3, g2/c**4, g3/c**6))

## The solution in original coordinates

We start by reminding ourselves of the solutions derived in *The general 2 mode case of the coherent coupler* notebook for modes in the original coordinates $u,v$:

In [60]:
Lams = [
    Eq(
        Lamda[0, m],
        -a[m] - gamma[m]*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2))
        + Sum(a[j, j]*gamma[j], (j, 1, 2))/2 - Sum(a[m, k]*gamma[k], (k, 1, 2)) + Sum(a[j]/2, (j, 1, 2))
    ),
    Eq(Lamda[1, m], Sum(a[m, k], (k, 1, 2)) - Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)))
]

u_sqrt_wp_z0_z1 = Eq(
    u(z, mu[j]),
    d[4]**Rational(-1, 4)*
    sqrt(wpp(z1, g2, g3))/
    sqrt((wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3)))/
    sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))
    *sigma(z - 2*z0 + mu[j], g2, g3)/(sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3))
    *exp(z*r[0, j] + log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] + epsilon[j])
)
v_sqrt_wp_z0_z1 = Eq(
    v(z, mu[j]),
    d[4]**Rational(-1, 4)*
    sqrt(wpp(z1, g2, g3))/
    sqrt((wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3)))/
    sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))
    *sigma(z - mu[j], g2, g3)/(sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3))
    *exp(-z*r[0, j] - log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] - epsilon[j])
)

uv_wp_z0z1muj = Eq(
    u(z, mu[j])*v(z, mu[j]),
    (wp(z - z0, g2, g3) - wp(-z0 + mu[j], g2, g3))
    *wpp(z1, g2, g3)/
    ((-wp(z1, g2, g3) + wp(z - z0, g2, g3))*(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sqrt(d[4]))
)

uv_wp_mu1_mu2_const = Eq(
    u(z, mu[1])*v(z, mu[1]) - u(z, mu[2])*v(z, mu[2]),
    (wp(-z0 + mu[1], g2, g3) - wp(-z0 + mu[2], g2, g3))*wpp(z1, g2, g3)/(
        (-wp(z1, g2, g3) + wp(-z0 + mu[1], g2, g3))*(-wp(z1, g2, g3) + wp(-z0 + mu[2], g2, g3))*sqrt(d[4])
    )
)

sum_r1j_1 = Eq(Sum(r[1, j], (j, 1, 2)), 1)
r1j_log_part = Eq(r[1, j], Lamda[1, j]/sqrt(d[4]))

u_sqrt_wp_z0_z1
v_sqrt_wp_z0_z1
uv_wp_mu1_mu2_const
uv_wp_z0z1muj
sum_r1j_1
r1j_log_part

for _ in Lams:
    _

Eq(u(z, mu[j]), sqrt(\wp'(z1, g2, g3))*sigma(z - 2*z0 + mu[j], g2, g3)*exp(z*r[0, j] + log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] + epsilon[j])/(sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)*d[4]**(1/4)))

Eq(v(z, mu[j]), sqrt(\wp'(z1, g2, g3))*sigma(z - mu[j], g2, g3)*exp(-z*r[0, j] - log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] - epsilon[j])/(sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)*d[4]**(1/4)))

Eq(u(z, mu[1])*v(z, mu[1]) - u(z, mu[2])*v(z, mu[2]), (wp(-z0 + mu[1], g2, g3) - wp(-z0 + mu[2], g2, g3))*\wp'(z1, g2, g3)/((-wp(z1, g2, g3) + wp(-z0 + mu[1], g2, g3))*(-wp(z1, g2, g3) + wp(-z0 + mu[2], g2, g3))*sqrt(d[4])))

Eq(u(z, mu[j])*v(z, mu[j]), (wp(z - z0, g2, g3) - wp(-z0 + mu[j], g2, g3))*\wp'(z1, g2, g3)/((-wp(z1, g2, g3) + wp(z - z0, g2, g3))*(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sqrt(d[4])))

Eq(Sum(r[1, j], (j, 1, 2)), 1)

Eq(r[1, j], Lamda[1, j]/sqrt(d[4]))

Eq(Lamda[0, m], -a[m] - gamma[m]*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)) + Sum(a[j, j]*gamma[j], (j, 1, 2))/2 - Sum(a[m, k]*gamma[k], (k, 1, 2)) + Sum(a[j]/2, (j, 1, 2)))

Eq(Lamda[1, m], Sum(a[m, k], (k, 1, 2)) - Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)))

### The original parameters

In [5]:
b_j_coeffs = [
    Eq(
        b[0], 
        (gamma[1] - gamma[2])**2*(Sum(a[j, j]/4, (j, 1, 2)) - Sum(a[j, k]/8, (j, 1, 2), (k, 1, 2))) 
        + a[0] + Sum(a[j]*gamma[j], (j, 1, 2))
    ),
    Eq(b[1], -Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j], (j, 1, 2))),
    Eq(b[2], Sum(a[j, k]/2, (j, 1, 2), (k, 1, 2)))
]

c_j_coeffs = [Eq(c[0], Product(gamma[j], (j, 1, 2))), Eq(c[1], 0), Eq(c[2], 1)]


d_j_coeffs = [
    Eq(d[0], b[0]**2 - 4*c[0]),
    Eq(d[1], 2*b[0]*b[1] - 4*c[1]),
    Eq(d[2], 2*b[0]*b[2] + b[1]**2 - 4*c[2]),
    Eq(d[3], 2*b[1]*b[2]),
    Eq(d[4], b[2]**2)
]

g2_dj = Eq(g2, d[0]*d[4] - d[1]*d[3]/4 + d[2]**2/12)
g3_dj = Eq(g3, d[0]*d[2]*d[4]/6 - d[0]*d[3]**2/16 - d[1]**2*d[4]/16 + d[1]*d[2]*d[3]/48 - d[2]**3/216)

for _ in b_j_coeffs:
    _
    
for _ in c_j_coeffs:
    _
    
for _ in d_j_coeffs:
    _

g2_dj
g3_dj

Eq(b[0], (gamma[1] - gamma[2])**2*(Sum(a[j, j]/4, (j, 1, 2)) - Sum(a[j, k]/8, (j, 1, 2), (k, 1, 2))) + a[0] + Sum(a[j]*gamma[j], (j, 1, 2)))

Eq(b[1], -Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j], (j, 1, 2)))

Eq(b[2], Sum(a[j, k]/2, (j, 1, 2), (k, 1, 2)))

Eq(c[0], Product(gamma[j], (j, 1, 2)))

Eq(c[1], 0)

...

Eq(d[2], 2*b[0]*b[2] + b[1]**2 - 4*c[2])

Eq(d[3], 2*b[1]*b[2])

Eq(d[4], b[2]**2)

Eq(g2, d[0]*d[4] - d[1]*d[3]/4 + d[2]**2/12)

Eq(g3, d[0]*d[2]*d[4]/6 - d[0]*d[3]**2/16 - d[1]**2*d[4]/16 + d[1]*d[2]*d[3]/48 - d[2]**3/216)

In [89]:
B_wpz1_z0_muj_to_d_lam1_gamj = Eq(
    wpp(z1, g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3)),
    -(gamma[j] - lamda[1])*sqrt(d[4])
)
C_wpz1_z0_muj_to_d_lam1_gamj = Eq(
    wpp(-z0 + mu[j], g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3)),
    -(b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)/(gamma[j] - lamda[1])
)
D_wpz1_z0_muj_to_d_lam1_gamj = Eq(
    wpp(z1, g2, g3)*wpp(-z0 + mu[j], g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3))**2,
    (b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)*sqrt(d[4])
)


wp_z1_lam_uv_j = Eq(
    wpp(z1, g2, g3)/((wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(d[4])),
    -lamda[1] - Sum(u(z, mu[j])*v(z, mu[j]), (j, 1, 2))/2
)
d4_lam_0 = Eq(0, d[0] + d[1]*lamda[1] + d[2]*lamda[1]**2 + d[3]*lamda[1]**3 + d[4]*lamda[1]**4)
d5_dj = Eq(d[5], d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)
wppz1_djs = Eq( 4*wpp(z1, g2, g3)/sqrt(d[4]), -d[5])

wp_z0_mu_j_d = Eq(
    wp(-z0 + mu[j], g2, g3),
    d[2]/12 + d[3]*lamda[1]/4 + d[4]*lamda[1]**2/2 - 
    (-d[1] - 2*d[2]*lamda[1] - 3*d[3]*lamda[1]**2 - 4*d[4]*lamda[1]**3)/(4*(gamma[j] - lamda[1]))
)
wpp_z0_mu_j_d = Eq(
    wpp(-z0 + mu[j], g2, g3),
    -(b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)
    *(d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)/(4*(gamma[j] - lamda[1])**2)
)
wp_z1_z0_mu_j_d5_lam = Eq(
    -(1/B_wpz1_z0_muj_to_d_lam1_gamj.lhs)*wppz1_djs.lhs*sqrt(d[4])/4, 
    -(1/B_wpz1_z0_muj_to_d_lam1_gamj.rhs)*wppz1_djs.rhs*sqrt(d[4])/4
)

wppz1_djs_sqrt = Eq(
    sqrt(fraction(wppz1_djs.lhs)[0])/sqrt(fraction(wppz1_djs.lhs)[1]), 
    I*varsigma*sqrt(-wppz1_djs.rhs)
)



B_wpz1_z0_muj_to_d_lam1_gamj
C_wpz1_z0_muj_to_d_lam1_gamj
D_wpz1_z0_muj_to_d_lam1_gamj
wp_z1_z0_mu_j_d5_lam
wp_z1_lam_uv_j
wppz1_djs
wppz1_djs_sqrt
wp_z0_mu_j_d
wpp_z0_mu_j_d

d4_lam_0
d5_dj


Eq(\wp'(z1, g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3)), (-gamma[j] + lamda[1])*sqrt(d[4]))

Eq(\wp'(-z0 + mu[j], g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3)), (-b[0] - b[1]*gamma[j] - b[2]*gamma[j]**2)/(gamma[j] - lamda[1]))

Eq(\wp'(z1, g2, g3)*\wp'(-z0 + mu[j], g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3))**2, (b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)*sqrt(d[4]))

Eq(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3), d[5]/(4*(-gamma[j] + lamda[1])))

Eq(\wp'(z1, g2, g3)/((wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(d[4])), -lamda[1] - Sum(u(z, mu[j])*v(z, mu[j]), (j, 1, 2))/2)

...

Eq(2*sqrt(\wp'(z1, g2, g3))/d[4]**(1/4), I*varsigma*sqrt(d[5]))

Eq(wp(-z0 + mu[j], g2, g3), d[2]/12 + d[3]*lamda[1]/4 + d[4]*lamda[1]**2/2 - (-d[1] - 2*d[2]*lamda[1] - 3*d[3]*lamda[1]**2 - 4*d[4]*lamda[1]**3)/(4*gamma[j] - 4*lamda[1]))

Eq(\wp'(-z0 + mu[j], g2, g3), (-b[0] - b[1]*gamma[j] - b[2]*gamma[j]**2)*(d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)/(4*(gamma[j] - lamda[1])**2))

Eq(0, d[0] + d[1]*lamda[1] + d[2]*lamda[1]**2 + d[3]*lamda[1]**3 + d[4]*lamda[1]**4)

Eq(d[5], d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)

We recall the equations from the original derivation of the solution in terms of $\rho, b_j, \gamma_j, \lambda_1$:

In [7]:
drhop_b = Eq(
    Derivative(rho(z), z)**2, 
    (rho(z)**2*b[2] + rho(z)*b[1] + b[0])**2 - 4*Product(-rho(z) + gamma[j], (j, 1, 2))
)
drhop_d = Eq(Derivative(rho(z), z)**2, Sum(rho(z)**j*d[j], (j, 0, 4)))

drho_2z = Eq(Derivative(rho(z), (z, 2)), Sum(j*rho(z)**(j-1)*d[j]/2, (j, 0, 4)))
drho_2z_b = Eq(
    Derivative(rho(z), (z, 2)),
    (2*rho(z)*b[2] + b[1])*(rho(z)**2*b[2] + rho(z)*b[1] + b[0]) + 2*(-2*rho(z) + gamma[1] + gamma[2])
)

gamma_lamda_bj = Eq(
    drhop_b.rhs.subs(rho(z), lamda[1]).doit()
    .subs(gamma[2], - gamma[1])
    .expand().collect(4*gamma[1]**2 - 4*lamda[1]**2, factor),
    0
)
gamma_lamda_bj_gam1 = Eq(gamma[1]**2, solve(gamma_lamda_bj, gamma[1]**2)[0])

drhop_b
drhop_d
drho_2z
drho_2z_b
gamma_lamda_bj_gam1

Eq(Derivative(rho(z), z)**2, (rho(z)**2*b[2] + rho(z)*b[1] + b[0])**2 - 4*Product(-rho(z) + gamma[j], (j, 1, 2)))

Eq(Derivative(rho(z), z)**2, Sum(rho(z)**j*d[j], (j, 0, 4)))

Eq(Derivative(rho(z), (z, 2)), Sum(j*rho(z)**(j - 1)*d[j]/2, (j, 0, 4)))

Eq(Derivative(rho(z), (z, 2)), (2*rho(z)*b[2] + b[1])*(rho(z)**2*b[2] + rho(z)*b[1] + b[0]) - 4*rho(z) + 2*gamma[1] + 2*gamma[2])

Eq(gamma[1]**2, -(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2/4 + lamda[1]**2)

In [8]:
bs_as_ds = [_.subs([_.args for _ in c_j_coeffs]) for _ in d_j_coeffs]

b0_d1 = Eq(b[0], solve(bs_as_ds[1], b[0])[0])
b2_d3 = Eq(b[2], solve(bs_as_ds[3], b[2])[0])
b1_sqrd_as_d = Eq(
    b[1]**2,
    solve(
        bs_as_ds[2]
        .subs([
            b0_d1.args, 
            (2*bs_as_ds[4].rhs/bs_as_ds[3].rhs, 2*bs_as_ds[4].lhs/bs_as_ds[3].lhs)
        ]),
        b[1]**2
    )[0]
)
b_to_d_subs = [
    b0_d1.args,
    b2_d3.args,
    b1_sqrd_as_d.args
]

for _ in bs_as_ds:
    _
print()
for _ in b_to_d_subs:
    Eq(*_)
    
def generate_lambda_reductions(max_power=12):
    """
    Generate substitutions reducing lamda**n (n >= 4)
    to expressions at most cubic in lamda.
    """
    subs = {}

    # base reduction: λ⁴
    subs[lamda[1]**4] = -(d[0] + d[1]*lamda[1] + d[2]*lamda[1]**2 + d[3]*lamda[1]**3) / d[4]

    # build higher powers
    for _n in range(5, max_power + 1):
        subs[lamda[1]**_n] = expand(lamda * subs[lamda[1]**(_n - 1)]).subs(subs)

    return subs

lamda_d_reductions = generate_lambda_reductions()

Eq(d[0], b[0]**2 - 4*Product(gamma[j], (j, 1, 2)))

Eq(d[1], 2*b[0]*b[1])

Eq(d[2], 2*b[0]*b[2] + b[1]**2 - 4)

Eq(d[3], 2*b[1]*b[2])

Eq(d[4], b[2]**2)

Eq(b[0], d[1]/(2*b[1]))

Eq(b[2], d[3]/(2*b[1]))

Eq(b[1]**2, -2*d[1]*d[4]/d[3] + d[2] + 4)

### Reorganising the original equations of motion

In [9]:
ajk_syms = [(a[2, 1], a[1, 2])]

uv_j_rho = Eq(u(z,mu[j])*v(z,mu[j]), gamma[j] - rho(z))

duvj = Eq(
    Derivative(u(z, mu[j])*v(z, mu[j]),z), 
    Product(v(z, mu[k]), (k, 1, 2)) - Product(u(z, mu[k]), (k, 1, 2))
)

uprodj = Eq(
    Product(u(z, mu[j]), (j, 1, 2)),
    -Derivative(u(z, mu[j])*v(z, mu[j]), z)/2 + a[0]/2 + 
    Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2))/2 + 
    Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2))/2
)
vprodj = Eq(uprodj.lhs + duvj.rhs.replace(k,j), uprodj.rhs + duvj.lhs.replace(k,j))
a_0_eq = Eq(uprodj.rhs + vprodj.rhs, uprodj.lhs + vprodj.lhs)

du_phase_mod_part = (a[j] + Sum(a[j,k]*u(z,mu[k])*v(z,mu[k]), (k,1,2)))*u(z,mu[j])
dv_phase_mod_part = -(a[j] + Sum(a[j,k]*u(z,mu[k])*v(z,mu[k]), (k,1,2)))*v(z,mu[j])

du_mixing_part = Product(v(z,mu[k]), (k,1,2))/v(z, mu[j])
dv_mixing_part = -Product(u(z,mu[k]), (k,1,2))/u(z, mu[j])

duj = Eq(diff(u(z,mu[j]),z), -du_phase_mod_part + du_mixing_part)
dvj = Eq(diff(v(z,mu[j]),z), (-dv_phase_mod_part).factor() + dv_mixing_part)
duj_subs = [duj.subs(j, _j + 1).args for _j in range(2)]
dvj_subs = [dvj.subs(j, _j + 1).args for _j in range(2)]
duj_dvj_subs = duj_subs + dvj_subs


assert (
    diff(solve(a_0_eq.doit(),a[0])[0],z).subs(duj_dvj_subs).doit().subs(ajk_syms).simplify() == 0
), 'a0 not conserved'

uv_j_rho

uprodj
vprodj
a_0_eq

duj
dvj
duvj

Eq(u(z, mu[j])*v(z, mu[j]), -rho(z) + gamma[j])

Eq(Product(u(z, mu[j]), (j, 1, 2)), -Derivative(u(z, mu[j])*v(z, mu[j]), z)/2 + a[0]/2 + Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2))/2 + Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2))/2)

Eq(Product(v(z, mu[j]), (j, 1, 2)), Derivative(u(z, mu[j])*v(z, mu[j]), z)/2 + a[0]/2 + Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2))/2 + Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2))/2)

Eq(a[0] + Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2)) + Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2)), Product(u(z, mu[j]), (j, 1, 2)) + Product(v(z, mu[j]), (j, 1, 2)))

Eq(Derivative(u(z, mu[j]), z), -(a[j] + Sum(u(z, mu[k])*v(z, mu[k])*a[j, k], (k, 1, 2)))*u(z, mu[j]) + Product(v(z, mu[k]), (k, 1, 2))/v(z, mu[j]))

Eq(Derivative(v(z, mu[j]), z), (a[j] + Sum(u(z, mu[k])*v(z, mu[k])*a[j, k], (k, 1, 2)))*v(z, mu[j]) - Product(u(z, mu[k]), (k, 1, 2))/u(z, mu[j]))

Eq(Derivative(u(z, mu[j])*v(z, mu[j]), z), -Product(u(z, mu[k]), (k, 1, 2)) + Product(v(z, mu[k]), (k, 1, 2)))

In [57]:
rho_integral = Eq(
    Integral(-rho(z) + rho(mu[j]),z),
    z*(zeta(-z0 + z1 + mu[j], g2, g3) + zeta(z0 + z1 - mu[j], g2, g3))/sqrt(d[4]) -
    log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/sqrt(d[4]) + C
)
zeta_z1_gamma_lam = (
    pw_to_zw_identity_one_sided
    .subs([(z,z1),(x,mu[j]-z0)])
    .subs([B_wpz1_z0_muj_to_d_lam1_gamj.args])
)
zeta_z1_gamma_lam = Eq(
    (-(zeta_z1_gamma_lam.rhs - 2*zeta(z1, g2, g3))/sqrt(d[4])),
    (-(zeta_z1_gamma_lam.lhs - 2*zeta(z1, g2, g3))/sqrt(d[4])).expand()
)

uv_integral_log_sigma = rho_integral.subs([
    (rho(z), solve(uv_j_rho, rho(z))[0]), 
    (gamma[j], rho(mu[j])),
    zeta_z1_gamma_lam.args
])

rho_integral
zeta_z1_gamma_lam
uv_integral_log_sigma

Eq(Integral(-rho(z) + rho(mu[j]), z), C + z*(zeta(-z0 + z1 + mu[j], g2, g3) + zeta(z0 + z1 - mu[j], g2, g3))/sqrt(d[4]) - log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/sqrt(d[4]))

Eq((zeta(-z0 + z1 + mu[j], g2, g3) + zeta(z0 + z1 - mu[j], g2, g3))/sqrt(d[4]), 2*zeta(z1, g2, g3)/sqrt(d[4]) + gamma[j] - lamda[1])

Eq(Integral(u(z, mu[j])*v(z, mu[j]), z), C + z*(2*zeta(z1, g2, g3)/sqrt(d[4]) + gamma[j] - lamda[1]) - log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))/sqrt(d[4]))

## The transfrom to polarisation dynamics

In [10]:
u_sqrt_wp_z0_z1
v_sqrt_wp_z0_z1

Eq(u(z, mu[j]), sqrt(\wp'(z1, g2, g3))*sigma(z - 2*z0 + mu[j], g2, g3)*exp(z*r[0, j] + log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] + epsilon[j])/(sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)*d[4]**(1/4)))

Eq(v(z, mu[j]), sqrt(\wp'(z1, g2, g3))*sigma(z - mu[j], g2, g3)*exp(-z*r[0, j] - log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] - epsilon[j])/(sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)*d[4]**(1/4)))

In [113]:
uhat_sigma_j = Eq(
    uhat(z, mu[j]), 
    sigma(z - 2*z0 + mu[j], g2, g3)/(sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3))
)
vhat_sigma_j = Eq(
    vhat(z, mu[j]), 
    sigma(z - mu[j], g2, g3)/(sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3))
)
uvhat_wp_j = Eq(
    uhat_sigma_j.lhs*vhat_sigma_j.lhs, 
    (uhat_sigma_j.rhs*vhat_sigma_j.rhs)
    .subs([
        (
            sigma_p_identity.subs([(x,z-z0), (y, mu[j] - z0)]).rhs,
            sigma_p_identity.subs([(x,z-z0), (y, mu[j] - z0)]).lhs
        )
    ])
)
uv_j_d5_gam_wp_z1 = Eq(
    uvhat_wp_j.lhs + wp_z1_z0_mu_j_d5_lam.rhs,
    uvhat_wp_j.rhs + wp_z1_z0_mu_j_d5_lam.lhs
)
uv_j_to_uvhat_j = uv_wp_z0z1muj.subs([
    (uvhat_wp_j.rhs, uvhat_wp_j.lhs),
    (uv_j_d5_gam_wp_z1.rhs, uv_j_d5_gam_wp_z1.lhs),
    wp_z1_z0_mu_j_d5_lam.args,
    wppz1_djs.args
])
uvhat_integral_log_sigma = uv_integral_log_sigma.replace(*uv_j_to_uvhat_j.args)
uvhat_integral_log_sigma = Eq(
    sqrt(d[4])*(
        uvhat_integral_log_sigma.lhs 
        - uvhat_integral_log_sigma.lhs 
        - uvhat_integral_log_sigma.rhs.args[2]
    ),
    sqrt(d[4])*(
        uvhat_integral_log_sigma.rhs 
        - uvhat_integral_log_sigma.lhs 
        - uvhat_integral_log_sigma.rhs.args[2]
    )
)
uvhat_integral_phiz = Eq(uvhat_integral_log_sigma.lhs, sqrt(d[4])*Integral(phi(z),z))


uhat_sigma_j
vhat_sigma_j
uvhat_wp_j
uv_j_d5_gam_wp_z1
uv_j_to_uvhat_j
uvhat_integral_log_sigma
uvhat_integral_phiz

Eq(uhat(z, mu[j]), sigma(z - 2*z0 + mu[j], g2, g3)/(sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)))

Eq(vhat(z, mu[j]), sigma(z - mu[j], g2, g3)/(sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)))

Eq(uhat(z, mu[j])*vhat(z, mu[j]), -wp(z - z0, g2, g3) + wp(-z0 + mu[j], g2, g3))

Eq(uhat(z, mu[j])*vhat(z, mu[j]) + d[5]/(4*(-gamma[j] + lamda[1])), wp(z1, g2, g3) - wp(z - z0, g2, g3))

Eq(u(z, mu[j])*v(z, mu[j]), (-gamma[j] + lamda[1])*uhat(z, mu[j])*vhat(z, mu[j])/(-uhat(z, mu[j])*vhat(z, mu[j]) - d[5]/(-4*gamma[j] + 4*lamda[1])))

Eq(log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3)), (C + z*(2*zeta(z1, g2, g3)/sqrt(d[4]) + gamma[j] - lamda[1]) - Integral((-gamma[j] + lamda[1])*uhat(z, mu[j])*vhat(z, mu[j])/(-uhat(z, mu[j])*vhat(z, mu[j]) - d[5]/(-4*gamma[j] + 4*lamda[1])), z))*sqrt(d[4]))

Eq(log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3)), sqrt(d[4])*Integral(phi(z), z))

In [114]:
to_hat_transform_subs =[
    (uvhat_wp_j.rhs, uvhat_wp_j.lhs),
    (uv_j_d5_gam_wp_z1.rhs, uv_j_d5_gam_wp_z1.lhs),
    wp_z1_z0_mu_j_d5_lam.args,
    wppz1_djs.args,
    (uhat_sigma_j.rhs, uhat_sigma_j.lhs),
    (vhat_sigma_j.rhs, vhat_sigma_j.lhs),
    wppz1_djs_sqrt.args,
    uvhat_integral_phiz.args,
    r1j_log_part.args
]
uj_as_uhatj = u_sqrt_wp_z0_z1.subs(to_hat_transform_subs)
vj_as_vhatj = v_sqrt_wp_z0_z1.subs(to_hat_transform_subs)

u_v_j_as_hats_subs = [
    uj_as_uhatj.subs(j,1).args,
    uj_as_uhatj.subs(j,2).args,
    vj_as_vhatj.subs(j,1).args,
    vj_as_vhatj.subs(j,2).args
]

uj_as_uhatj
vj_as_vhatj

Eq(u(z, mu[j]), I*varsigma*uhat(z, mu[j])*exp(z*r[0, j] + Lamda[1, j]*Integral(phi(z), z) + epsilon[j])*sqrt(d[5])/(sqrt(d[5]/(-gamma[j] + lamda[1]))*sqrt(uhat(z, mu[j])*vhat(z, mu[j]) + d[5]/(-4*gamma[j] + 4*lamda[1]))))

Eq(v(z, mu[j]), I*varsigma*vhat(z, mu[j])*exp(-z*r[0, j] - Lamda[1, j]*Integral(phi(z), z) - epsilon[j])*sqrt(d[5])/(sqrt(d[5]/(-gamma[j] + lamda[1]))*sqrt(uhat(z, mu[j])*vhat(z, mu[j]) + d[5]/(-4*gamma[j] + 4*lamda[1]))))

In [115]:
duj
dvj

Eq(Derivative(u(z, mu[j]), z), -(a[j] + Sum(u(z, mu[k])*v(z, mu[k])*a[j, k], (k, 1, 2)))*u(z, mu[j]) + Product(v(z, mu[k]), (k, 1, 2))/v(z, mu[j]))

Eq(Derivative(v(z, mu[j]), z), (a[j] + Sum(u(z, mu[k])*v(z, mu[k])*a[j, k], (k, 1, 2)))*v(z, mu[j]) - Product(u(z, mu[k]), (k, 1, 2))/u(z, mu[j]))

In [116]:
du_dv_js = [
    duj.doit().subs(j,1),
    duj.doit().subs(j,2),
    dvj.doit().subs(j,1),
    dvj.doit().subs(j,2)
]

for _ in du_dv_js:
    _

Eq(Derivative(u(z, mu[1]), z), -(u(z, mu[1])*v(z, mu[1])*a[1, 1] + u(z, mu[2])*v(z, mu[2])*a[1, 2] + a[1])*u(z, mu[1]) + v(z, mu[2]))

Eq(Derivative(u(z, mu[2]), z), -(u(z, mu[1])*v(z, mu[1])*a[2, 1] + u(z, mu[2])*v(z, mu[2])*a[2, 2] + a[2])*u(z, mu[2]) + v(z, mu[1]))

Eq(Derivative(v(z, mu[1]), z), (u(z, mu[1])*v(z, mu[1])*a[1, 1] + u(z, mu[2])*v(z, mu[2])*a[1, 2] + a[1])*v(z, mu[1]) - u(z, mu[2]))

Eq(Derivative(v(z, mu[2]), z), (u(z, mu[1])*v(z, mu[1])*a[2, 1] + u(z, mu[2])*v(z, mu[2])*a[2, 2] + a[2])*v(z, mu[2]) - u(z, mu[1]))

In [126]:
dhat1 = du_dv_js[0].subs(u_v_j_as_hats_subs).doit()

dhat1 = Eq(dhat1.lhs.simplify(), sum(_.simplify() for _ in dhat1.rhs.expand().args))

In [127]:
Eq(
    diff(uhat(z, mu[1]),z), 
    solve(dhat1, diff(uhat(z, mu[1]),z))[0]
)

KeyboardInterrupt: 